In [0]:
DROP SCHEMA IF EXISTS mini_project.gold CASCADE

In [0]:
CREATE SCHEMA IF NOT EXISTS mini_project.gold
  MANAGED LOCATION 's3://db-lakehouse-s3/lakehouse/gold'
  

In [0]:
DROP TABLE IF EXISTS mini_project.gold.dim_date;
DROP TABLE IF EXISTS mini_project.gold.dim_company;
DROP TABLE IF EXISTS mini_project.gold.dim_department;
DROP TABLE IF EXISTS mini_project.gold.dim_account;
DROP TABLE IF EXISTS mini_project.gold.dim_employee;
DROP TABLE IF EXISTS mini_project.gold.dim_position;
DROP TABLE IF EXISTS mini_project.gold.fact_general_ledger;
DROP TABLE IF EXISTS mini_project.gold.fact_payroll;

##dim date

In [0]:
CREATE OR REPLACE TABLE mini_project.gold.dim_date AS
WITH date_range AS (
  SELECT explode(sequence(
    to_date('2020-01-01'),
    to_date('2030-12-31'),
    interval 1 day
  )) AS date_key
)
SELECT 
  date_key,
  year(date_key) AS year,
  quarter(date_key) AS quarter,
  month(date_key) AS month,
  dayofmonth(date_key) AS day,
  dayofweek(date_key) AS day_of_week,
  dayofyear(date_key) AS day_of_year,
  weekofyear(date_key) AS week_of_year,
  date_format(date_key, 'MMMM') AS month_name,
  date_format(date_key, 'EEEE') AS day_name,
  date_format(date_key, 'yyyy-MM') AS year_month,
  concat('Q', quarter(date_key), ' ', year(date_key)) AS quarter_year,
  CASE WHEN dayofweek(date_key) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  CASE WHEN month(date_key) IN (1,2,3) THEN 'Q1'
       WHEN month(date_key) IN (4,5,6) THEN 'Q2'
       WHEN month(date_key) IN (7,8,9) THEN 'Q3'
       ELSE 'Q4' END AS fiscal_quarter,
  date_sub(date_key, dayofmonth(date_key) - 1) AS first_day_of_month,
  last_day(date_key) AS last_day_of_month
FROM date_range
ORDER BY date_key;

SELECT 'Date dimension created with ' || COUNT(*) || ' rows' AS status FROM mini_project.gold.dim_date;

##dim company

In [0]:
CREATE OR REPLACE TABLE mini_project.gold.dim_company AS
SELECT 
  company_id,
  company_name,
  industry,
  country,
  established_date,
  is_active,
  current_timestamp() AS last_updated
FROM mini_project.silver.silver_company
WHERE is_active = TRUE;

SELECT 'Company dimension created with ' || COUNT(*) || ' companies' AS status FROM mini_project.gold.dim_company;

##dim dept

In [0]:
CREATE OR REPLACE TABLE mini_project.gold.dim_department AS
SELECT 
  department_id,
  department_code,
  department_name,
  cost_center,
  current_timestamp() AS last_updated
FROM mini_project.silver.silver_departments;

SELECT 'Department dimension created with ' || COUNT(*) || ' departments' AS status FROM mini_project.gold.dim_department;

##dim account

In [0]:
CREATE OR REPLACE TABLE mini_project.gold.dim_account AS
SELECT 
  account_id,
  account_code,
  account_name,
  account_type,
  category,
  is_active,
  
  CASE 
    WHEN account_id BETWEEN 4000 AND 4999 THEN 'Revenue'
    WHEN account_id BETWEEN 5000 AND 5999 THEN 'Cost of Sales'
    WHEN account_id BETWEEN 6000 AND 6099 THEN 'Personnel Expense'
    WHEN account_id BETWEEN 6100 AND 6199 THEN 'Occupancy Expense'
    WHEN account_id BETWEEN 6200 AND 6299 THEN 'Technology Expense'
    WHEN account_id BETWEEN 6300 AND 6399 THEN 'Marketing Expense'
    WHEN account_id BETWEEN 6400 AND 6499 THEN 'Travel & Entertainment'
    WHEN account_id BETWEEN 6500 AND 6599 THEN 'Professional Services'
    WHEN account_id BETWEEN 6600 AND 6699 THEN 'Communication Expense'
    WHEN account_id BETWEEN 6700 AND 6799 THEN 'Financial Expense'
    WHEN account_id BETWEEN 6800 AND 6899 THEN 'Depreciation & Amortization'
    WHEN account_id BETWEEN 6900 AND 6999 THEN 'Other Operating Expense'
    WHEN account_id BETWEEN 7000 AND 7099 THEN 'R&D Expense'
    WHEN account_id BETWEEN 7100 AND 7199 THEN 'Tax Expense'
    ELSE 'Other'
  END AS pl_section,
  CASE 
    WHEN account_id BETWEEN 4000 AND 4999 THEN 1
    WHEN account_id BETWEEN 5000 AND 5999 THEN 2
    WHEN account_id BETWEEN 6000 AND 7999 THEN 3
    ELSE 4
  END AS pl_order,
  current_timestamp() AS last_updated
FROM mini_project.silver.silver_accounts
WHERE is_active = TRUE;

select *  from  mini_project.gold.dim_account

##dim employee

In [0]:
CREATE OR REPLACE TABLE mini_project.gold.dim_employee AS
SELECT 
  employee_id,
  employee_code,
  first_name,
  last_name,
  concat(first_name, ' ', last_name) AS full_name,
  email,
  department_id,
  company_id,
  position,
  join_date,
  termination_date,
  base_salary,
  is_active,
  CASE 
    WHEN termination_date IS NULL OR termination_date > current_date() THEN TRUE 
    ELSE FALSE 
  END AS is_currently_active,
  current_timestamp() AS last_updated
FROM mini_project.silver.silver_employee;

SELECT 'Employee dimension created with ' || COUNT(*) || ' employees' AS status FROM mini_project.gold.dim_employee;

##dim position

In [0]:
CREATE OR REPLACE TABLE mini_project.gold.dim_position AS
SELECT DISTINCT
  row_number() OVER (ORDER BY position) AS position_id,
  position AS position_name,
  CASE 
    WHEN position LIKE '%Manager%' OR position LIKE '%Director%' OR position LIKE '%VP%' OR position LIKE '%Chief%' THEN 'Management'
    WHEN position LIKE '%Lead%' OR position LIKE '%Senior%' OR position LIKE '%Principal%' THEN 'Senior'
    WHEN position LIKE '%Junior%' OR position LIKE '%Associate%' THEN 'Junior'
    ELSE 'Mid-Level'
  END AS position_level,
  current_timestamp() AS last_updated
FROM mini_project.silver.silver_employee
WHERE position IS NOT NULL;

SELECT 'Position dimension created with ' || COUNT(*) || ' positions' AS status FROM mini_project.gold.dim_position;

##fact gl

In [0]:
CREATE OR REPLACE TABLE mini_project.gold.fact_general_ledger AS
SELECT 
  gl.gl_id,
  gl.entry_date,
  gl.posting_date,
  gl.fiscal_year,
  gl.fiscal_period,
  gl.company_id,
  c.company_name,
  gl.account_id,
  a.account_code,
  a.account_name,
  a.account_type,
  a.pl_section,
  gl.department_id,
  d.department_name,
  d.cost_center,
  gl.transaction_type,
  gl.reference_number,
  gl.description,
  gl.debit_amount,
  gl.credit_amount,
  CASE 
    WHEN a.account_type IN ('Revenue', 'Contra Asset', 'Liability', 'Equity') THEN gl.credit_amount - gl.debit_amount
    ELSE gl.debit_amount - gl.credit_amount
  END AS net_amount,
  gl.status,
  gl.created_by,
  gl.approved_by,
  current_timestamp() AS last_updated
FROM mini_project.silver.silver_general_ledgers gl
INNER JOIN mini_project.gold.dim_company c ON gl.company_id = c.company_id
INNER JOIN mini_project.gold.dim_account a ON gl.account_id = a.account_id
INNER JOIN mini_project.gold.dim_department d ON gl.department_id = d.department_id
WHERE gl.status IN ('Posted', 'Pending');

SELECT 'General ledger fact table created with ' || COUNT(*) || ' transactions' AS status FROM mini_project.gold.fact_general_ledger;

##fact payroll

In [0]:
CREATE OR REPLACE TABLE mini_project.gold.fact_payroll AS
SELECT 
  p.payroll_id,
  p.employee_id,
  e.employee_code,
  e.full_name,
  e.position,
  p.company_id,
  c.company_name,
  p.department_id,
  d.department_name,
  d.cost_center,
  p.pay_period_start,
  p.pay_period_end,
  p.pay_date,
  -- Use base_salary from employee dimension (ignore silver's gross_salary)
  e.base_salary AS gross_salary,
  p.bonus,
  p.overtime_pay,
  p.commission,
  p.allowances,
  -- Total compensation: base_salary + bonus + overtime + commission + allowances
  e.base_salary + p.bonus + p.overtime_pay + 
  p.commission + p.allowances AS total_compensation,
  -- Total deductions
  p.tax_deduction + p.social_security + p.health_insurance + 
  p.retirement_contribution + p.other_deductions AS total_deductions,
  p.tax_deduction,
  p.social_security,
  p.health_insurance,
  p.retirement_contribution,
  p.other_deductions,
  -- CALCULATE net_salary correctly (don't copy from silver!)
  e.base_salary + p.bonus + p.overtime_pay + 
  p.commission + p.allowances - 
  (p.tax_deduction + p.social_security + p.health_insurance + 
   p.retirement_contribution + p.other_deductions) AS net_salary,
  p.payment_method,
  p.status,
  -- Calculated metrics using base_salary
  (p.overtime_pay / e.base_salary) * 100  AS overtime_to_base_ratio,
  current_timestamp() AS last_updated
FROM mini_project.silver.silver_payroll p
INNER JOIN mini_project.gold.dim_employee e ON p.employee_id = e.employee_id
INNER JOIN mini_project.gold.dim_company c ON p.company_id = c.company_id
INNER JOIN mini_project.gold.dim_department d ON p.department_id = d.department_id;

select *  from mini_project.gold.fact_payroll

##datacube - unified analytics layer

In [0]:
select * from mini_project.silver.silver_payroll where employee_id = 11228;